# Historical Price & Return Analysis Framework
### Consecutive streak detection · Cumulative threshold analysis · Interactive Plotly dashboard

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with
# `pip install -e .`, so `from src.tools...` resolves from any working
# directory. On Colab the same editable install is done here against the
# copy of the repo on Drive.
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    !pip install -q -e /content/drive/MyDrive/github/quant_dev

print(f'Running in Colab: {IN_COLAB}')

In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Analysis parameters (edit freely) ───────────────────────────────────
TRADE_DAYS     = 250                 # trading days per year, used for all rolling/annualized stats
WIN_THRESHOLD  =  0.5                # min daily return (%) to count as a 'win'
LOSS_THRESHOLD = -0.5                # max daily return (%) to count as a 'loss'
WINDOWS        = [2, 3, 5]           # consecutive-day streak windows
CUM_THRESHOLDS = [0.5, 1.0, 2.0]     # cumulative return thresholds (%)
ROLL_WINDOWS   = [1, 2, 3, 4, 5, 10, TRADE_DAYS]   # holding-period windows for rolling avg/std stats

# ── Price data: simulated or real (Yahoo Finance) ────────────────────────
DATA_SOURCE = 'simulated'                             # 'simulated' or 'yahoo'
TICKER      = 'AAPL'                                  # change as needed
START_DATE  = '2020-01-01'                            # used by both sources
END_DATE    = dt.date.today().strftime('%Y-%m-%d')    # defaults to today

if DATA_SOURCE == 'yahoo':
    # pip install yfinance
    import yfinance as yf
    raw = yf.download(TICKER, start=START_DATE, end=END_DATE)
    if isinstance(raw.columns, pd.MultiIndex):         # yfinance can return a (Price, Ticker) MultiIndex
        raw.columns = raw.columns.get_level_values(0)
    df = raw[['Close']].rename(columns={'Close': 'price'}).reset_index()
    df = df.rename(columns={'Date': 'date'})
    df['return_pct'] = df['price'].pct_change() * 100
    df = df.dropna().reset_index(drop=True)

else:  # 'simulated'
    np.random.seed(42)
    dates = pd.bdate_range(start=START_DATE, end=END_DATE)
    daily_returns = np.random.normal(loc=0.04, scale=1.2, size=len(dates))   # % per day
    prices = 150 * np.cumprod(1 + daily_returns / 100)
    df = pd.DataFrame({
        'date':        dates,
        'price':       prices,
        'return_pct':  daily_returns
    })

df['date'] = pd.to_datetime(df['date'])
print(f'Data source: {DATA_SOURCE}   |   Loaded {len(df)} trading days   |   '
      f'Avg return: {df.return_pct.mean():.3f}%   |   '
      f'Std: {df.return_pct.std():.3f}%')
df.head()

# Rolling Window Statistics
### 250-trading-day rolling average & standard deviation, by holding-period window

In [ ]:
# ── Rolling window statistics (250-trading-day rolling avg & std) ───────
# For each holding period `d`, compute the d-day cumulative return
# (`PCT Change {d}`), then the TRADE_DAYS-day rolling mean (`{d} Av`) and
# rolling std (`{d} STD`) of that d-day return series. Returns are converted
# from % (e.g. 0.04) to decimal fraction (0.0004).
# See the "Analysis parameters" cell above for TRADE_DAYS / ROLL_WINDOWS.
for d in ROLL_WINDOWS:
    if d == 1:
        df['PCT Change 1']     = df['return_pct'] / 100
        df['PCT Change 1 Av']  = df['PCT Change 1'].rolling(TRADE_DAYS).mean()
        df['PCT Change 1 STD'] = df['PCT Change 1'].rolling(TRADE_DAYS).std()
    else:
        df[f'PCT Change {d}']     = df['PCT Change 1'].rolling(d).sum()
        df[f'PCT Change {d} Av']  = df[f'PCT Change {d}'].rolling(TRADE_DAYS).mean()
        df[f'PCT Change {d} STD'] = df[f'PCT Change {d}'].rolling(TRADE_DAYS).std()

    df['PCT Change Annualized']     = df['PCT Change 1'].rolling(TRADE_DAYS).sum()
    df['PCT Change Annualized STD'] = df['PCT Change 1 STD'] * TRADE_DAYS ** 0.5

print(f'Rolling stats computed for windows: {ROLL_WINDOWS} ({TRADE_DAYS}-day rolling avg/std)')
df[[c for c in df.columns if c.startswith('PCT Change')]].tail()

In [ ]:
# ── Latest snapshot: annualized & daily vol/mean ─────────────────────────
print('=== Annualized (latest 250d) ===')
print(f"Return:     {df['PCT Change Annualized'].iloc[-1]:.2%}")
print(f"Volatility: {df['PCT Change Annualized STD'].iloc[-1]:.2%}")
print()
print('=== Daily (latest 250d rolling) ===')
print(f"Avg return: {df['PCT Change 1 Av'].iloc[-1]:.4%}")
print(f"Std dev:    {df['PCT Change 1 STD'].iloc[-1]:.4%}")

In [ ]:
# ── Chart: 250d Rolling Average Return, by holding-period window ────────
fig_roll_av = go.Figure()
for d in [1, 2, 3, 4, 5, 10]:
    fig_roll_av.add_trace(go.Scatter(
        x=df['date'], y=df[f'PCT Change {d} Av'] * 100,
        mode='lines', name=f'{d}d'
    ))
fig_roll_av.update_layout(
    title='250-Day Rolling Average Return by Holding Period',
    xaxis_title='Date', yaxis_title='Rolling Avg Return (%)',
    height=400, plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=50, r=20, t=80, b=40)
)
fig_roll_av.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig_roll_av.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig_roll_av.show()

In [ ]:
# ── Chart: 250d Rolling Volatility (Std Dev), by holding-period window ──
fig_roll_std = go.Figure()
for d in [1, 2, 3, 4, 5, 10]:
    fig_roll_std.add_trace(go.Scatter(
        x=df['date'], y=df[f'PCT Change {d} STD'] * 100,
        mode='lines', name=f'{d}d'
    ))
fig_roll_std.add_trace(go.Scatter(
    x=df['date'], y=df['PCT Change Annualized STD'] * 100,
    mode='lines', name='Annualized', line=dict(color='black', dash='dash')
))
fig_roll_std.update_layout(
    title='250-Day Rolling Volatility (Std Dev) by Holding Period',
    xaxis_title='Date', yaxis_title='Rolling Std Dev (%)',
    height=400, plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=50, r=20, t=80, b=40)
)
fig_roll_std.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig_roll_std.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig_roll_std.show()

In [ ]:
# ── Streak detection ─────────────────────────────────────────────────────
def detect_streaks(df, win_thr, loss_thr, windows):
    results = {}
    rets = df['return_pct'].values
    dates = df['date'].values

    for w in windows:
        wins, losses = [], []
        for i in range(w - 1, len(rets)):
            window_rets = rets[i - w + 1 : i + 1]
            avg = window_rets.mean()
            entry = {
                'start':      pd.Timestamp(dates[i - w + 1]),
                'end':        pd.Timestamp(dates[i]),
                'returns':    window_rets.tolist(),
                'avg_return': round(float(avg), 4),
                'min_return': round(float(window_rets.min()), 4),
                'max_return': round(float(window_rets.max()), 4),
            }
            if all(r >= win_thr for r in window_rets):
                wins.append(entry)
            if all(r <= loss_thr for r in window_rets):
                losses.append(entry)
        results[w] = {'wins': wins, 'losses': losses}
    return results

streaks = detect_streaks(df, WIN_THRESHOLD, LOSS_THRESHOLD, WINDOWS)

# Summary table
streak_summary = []
total_windows = len(df)
for w in WINDOWS:
    nw = len(streaks[w]['wins'])
    nl = len(streaks[w]['losses'])
    streak_summary.append({
        'Window': f'{w}d',
        'Win Streaks': nw,
        'Win Freq %': round(nw / total_windows * 100, 2),
        'Loss Streaks': nl,
        'Loss Freq %': round(nl / total_windows * 100, 2),
        'Avg Win Ret %': round(np.mean([s['avg_return'] for s in streaks[w]['wins']]) if nw else 0, 3),
        'Avg Loss Ret %': round(np.mean([s['avg_return'] for s in streaks[w]['losses']]) if nl else 0, 3),
    })

streak_df = pd.DataFrame(streak_summary)
print('Streak summary')
streak_df

In [ ]:
# ── Cumulative return threshold analysis ─────────────────────────────────
def analyze_cumulative(df, windows, thresholds):
    rets = df['return_pct'].values
    dates = df['date'].values
    results = {}

    for w in windows:
        results[w] = {}
        for thr in thresholds:
            hits = []
            for i in range(w - 1, len(rets)):
                window_rets = rets[i - w + 1 : i + 1]
                cum_ret = (np.prod(1 + window_rets / 100) - 1) * 100
                if cum_ret >= thr:
                    hits.append({
                        'end_date': pd.Timestamp(dates[i]),
                        'cum_return': round(float(cum_ret), 4)
                    })
            n_windows = len(rets) - w + 1
            results[w][thr] = {
                'count': len(hits),
                'frequency': round(len(hits) / n_windows * 100, 2),
                'hits': hits
            }
    return results

cum_analysis = analyze_cumulative(df, WINDOWS, CUM_THRESHOLDS)

# Summary table
cum_rows = []
for w in WINDOWS:
    row = {'Window': f'{w}d'}
    for thr in CUM_THRESHOLDS:
        d = cum_analysis[w][thr]
        row[f'≥{thr}% Count']  = d['count']
        row[f'≥{thr}% Freq %'] = d['frequency']
    cum_rows.append(row)

cum_df = pd.DataFrame(cum_rows)
print('Cumulative return threshold summary')
cum_df

In [ ]:
# ── Chart 1: Price & Daily Return ───────────────────────────────────────
fig1 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=(f'{TICKER} Price', 'Daily Return (%)')
)

fig1.add_trace(go.Scatter(
    x=df['date'], y=df['price'],
    name='Price', line=dict(color='#378ADD', width=1.5)
), row=1, col=1)

colors_ret = ['#1D9E75' if r >= 0 else '#D85A30' for r in df['return_pct']]
fig1.add_trace(go.Bar(
    x=df['date'], y=df['return_pct'],
    name='Daily Return %', marker_color=colors_ret, opacity=0.8
), row=2, col=1)

fig1.add_hline(y=WIN_THRESHOLD,  line_dash='dash', line_color='#1D9E75', opacity=0.6, row=2, col=1)
fig1.add_hline(y=LOSS_THRESHOLD, line_dash='dash', line_color='#D85A30', opacity=0.6, row=2, col=1)

fig1.update_layout(
    height=500, title_text=f'{TICKER} — Historical Price & Daily Returns',
    showlegend=False, plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=50, r=20, t=60, b=40)
)
fig1.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig1.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig1.show()

In [ ]:
# ── Chart 2: Return Distribution ────────────────────────────────────────
bins = np.arange(-5, 5.5, 0.5)
hist_vals, bin_edges = np.histogram(df['return_pct'], bins=bins)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bar_colors = ['#1D9E75' if c >= 0 else '#D85A30' for c in bin_centers]

fig2 = go.Figure(go.Bar(
    x=bin_centers, y=hist_vals,
    marker_color=bar_colors, opacity=0.85,
    hovertemplate='Return: %{x:.1f}%<br>Days: %{y}<extra></extra>'
))
fig2.add_vline(x=WIN_THRESHOLD,  line_dash='dash', line_color='#1D9E75', annotation_text='Win thr')
fig2.add_vline(x=LOSS_THRESHOLD, line_dash='dash', line_color='#D85A30', annotation_text='Loss thr')
fig2.update_layout(
    title='Daily Return Distribution',
    xaxis_title='Return (%)', yaxis_title='Frequency (days)',
    height=350, plot_bgcolor='white', paper_bgcolor='white',
    bargap=0.05, margin=dict(l=50, r=20, t=60, b=40)
)
fig2.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.show()

In [ ]:
# ── Chart 3: Consecutive Win & Loss Streak Frequency ────────────────────
windows_labels = [f'{w}d' for w in WINDOWS]
win_counts  = [len(streaks[w]['wins'])   for w in WINDOWS]
loss_counts = [len(streaks[w]['losses']) for w in WINDOWS]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    name=f'Win streaks (each day ≥{WIN_THRESHOLD}%)',
    x=windows_labels, y=win_counts,
    marker_color='#1D9E75', text=win_counts, textposition='outside'
))
fig3.add_trace(go.Bar(
    name=f'Loss streaks (each day ≤{LOSS_THRESHOLD}%)',
    x=windows_labels, y=loss_counts,
    marker_color='#D85A30', text=loss_counts, textposition='outside'
))
fig3.update_layout(
    title='Consecutive Daily Win & Loss Streaks by Window',
    xaxis_title='Window', yaxis_title='Count',
    barmode='group', height=400,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=50, r=20, t=80, b=40)
)
fig3.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig3.show()

In [ ]:
# ── Chart 4: Streak Frequency as % of Trading Days ─────────────────────
win_freq  = [round(len(streaks[w]['wins'])   / len(df) * 100, 2) for w in WINDOWS]
loss_freq = [round(len(streaks[w]['losses']) / len(df) * 100, 2) for w in WINDOWS]

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    name='Win freq %', x=windows_labels, y=win_freq,
    marker_color='#1D9E75',
    text=[f'{v}%' for v in win_freq], textposition='outside'
))
fig4.add_trace(go.Bar(
    name='Loss freq %', x=windows_labels, y=loss_freq,
    marker_color='#D85A30',
    text=[f'{v}%' for v in loss_freq], textposition='outside'
))
fig4.update_layout(
    title='Streak Frequency as % of All Trading Days',
    xaxis_title='Window', yaxis_title='Frequency (%)',
    barmode='group', height=400,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=50, r=20, t=80, b=40)
)
fig4.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig4.show()

In [ ]:
# ── Chart 5: Cumulative Return Thresholds — Heatmap ─────────────────────
z_count = [[cum_analysis[w][t]['count'] for t in CUM_THRESHOLDS] for w in WINDOWS]
z_freq  = [[cum_analysis[w][t]['frequency'] for t in CUM_THRESHOLDS] for w in WINDOWS]

fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Count of windows exceeding threshold', 'Frequency (%) of windows exceeding threshold')
)

fig5.add_trace(go.Heatmap(
    z=z_count,
    x=[f'≥{t}%' for t in CUM_THRESHOLDS],
    y=[f'{w}d' for w in WINDOWS],
    colorscale='Blues', text=z_count, texttemplate='%{text}',
    showscale=True, colorbar=dict(x=0.45, len=0.8)
), row=1, col=1)

fig5.add_trace(go.Heatmap(
    z=z_freq,
    x=[f'≥{t}%' for t in CUM_THRESHOLDS],
    y=[f'{w}d' for w in WINDOWS],
    colorscale='Purples', text=z_freq, texttemplate='%{text}%',
    showscale=True, colorbar=dict(x=1.02, len=0.8)
), row=1, col=2)

fig5.update_layout(
    title='Cumulative Return Threshold Analysis (2d / 3d / 5d windows)',
    height=320, plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=60, r=60, t=80, b=40)
)
fig5.show()

In [ ]:
# ── Chart 6: Cumulative Threshold Counts — Grouped Bar ──────────────────
palette = ['#378ADD', '#7F77DD', '#D4537E']
fig6 = go.Figure()
for thr, color in zip(CUM_THRESHOLDS, palette):
    counts = [cum_analysis[w][thr]['count'] for w in WINDOWS]
    fig6.add_trace(go.Bar(
        name=f'≥{thr}%',
        x=windows_labels, y=counts,
        marker_color=color,
        text=counts, textposition='outside'
    ))

fig6.update_layout(
    title='Windows where Cumulative Return Exceeded Threshold',
    xaxis_title='Rolling Window', yaxis_title='Count',
    barmode='group', height=420,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Threshold', orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=50, r=20, t=80, b=40)
)
fig6.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig6.show()

In [ ]:
# ── Chart 7: Streak Timeline — when streaks occurred ────────────────────
STREAK_WINDOW = 2   # pick 2, 3, or 5

fig7 = go.Figure()
fig7.add_trace(go.Scatter(
    x=df['date'], y=df['return_pct'],
    mode='lines', name='Daily return',
    line=dict(color='#B4B2A9', width=1), opacity=0.7
))

for s in streaks[STREAK_WINDOW]['wins']:
    fig7.add_vrect(
        x0=s['start'], x1=s['end'],
        fillcolor='#1D9E75', opacity=0.15, line_width=0
    )

for s in streaks[STREAK_WINDOW]['losses']:
    fig7.add_vrect(
        x0=s['start'], x1=s['end'],
        fillcolor='#D85A30', opacity=0.15, line_width=0
    )

fig7.add_hline(y=WIN_THRESHOLD,  line_dash='dot', line_color='#1D9E75', opacity=0.8)
fig7.add_hline(y=LOSS_THRESHOLD, line_dash='dot', line_color='#D85A30', opacity=0.8)

fig7.update_layout(
    title=f'{STREAK_WINDOW}-day streak timeline (green=win, red=loss)',
    xaxis_title='Date', yaxis_title='Daily Return (%)',
    height=380, plot_bgcolor='white', paper_bgcolor='white',
    showlegend=False, margin=dict(l=50, r=20, t=60, b=40)
)
fig7.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig7.show()

In [ ]:
# ── Export summary tables to CSV (optional) ─────────────────────────────
streak_df.to_csv('streak_summary.csv', index=False)
cum_df.to_csv('cumulative_summary.csv', index=False)
df.to_csv('rolling_stats.csv', index=False)
print('streak_summary.csv, cumulative_summary.csv, and rolling_stats.csv saved.')
print()
print('=== Streak Summary ===')
print(streak_df.to_string(index=False))
print()
print('=== Cumulative Summary ===')
print(cum_df.to_string(index=False))